# Story 1: planetary-dominated reference eddies

Select stable deep-water AE and CE examples that establish the background polarity-dependent tilt. Under the corrected signed-PV convention, AEs are expected to tilt along signed $\nabla PV$ and CEs opposite it. These are reference states, not examples of every offshore eddy.

In [ ]:
from pathlib import Path
import sys
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
HERE = Path.cwd().resolve()
ROOT = next((p for p in (HERE, *HERE.parents) if (p / 'seacofs_tilt_tools.py').exists()), None)
if ROOT is None: raise FileNotFoundError('Run from seacofs_eddy_tilt_analysis or a subfolder.')
for path in (ROOT, ROOT / 'case_studies'):
    if str(path) not in sys.path: sys.path.insert(0, str(path))
import seacofs_tilt_tools as tilt
from paper_case_study_tools import PaperCaseConfig, plot_paper_case, rank_planetary_reference_cases, select_ranked_cases
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 40)

## Load and classify

Core-mean bathymetry and gradients are used. The main regime requires planetary dominance by at least 2:1 and depth of at least 3,000 m. Bearings with tilt distance below 5 km are excluded from directional ranking.

In [ ]:
config = PaperCaseConfig()
paths = tilt.Paths()
grid = tilt.load_grid(paths.grid, paths.z_r)
df_eddies, _ = tilt.load_tilt_tables(paths)
df_eddies = tilt.add_region_labels(df_eddies, grid)
df_eddies = tilt.add_pv_gradient_terms(df_eddies, grid, core_mean=True)
df_case, ranking = rank_planetary_reference_cases(df_eddies, config)
display(ranking.groupby('Cyc')['eligible'].agg(candidates='size', eligible='sum'))

## Population context

The case figures should accompany this complete-population distribution. A low polarity-aware error means AE alignment or CE opposition, as appropriate.

In [ ]:
use = df_case[df_case.planetary_regime & df_case.direction_valid]
bins = np.arange(0, 181, 10)
fig, ax = plt.subplots(figsize=(7, 4.5))
for cyc, color in [('AE', 'firebrick'), ('CE', 'royalblue')]:
    values = use.loc[use.Cyc.eq(cyc), 'preference_error_deg']
    ax.hist(values, bins=bins, density=True, histtype='step', lw=2, color=color, label=f'{cyc} (n={len(values):,})')
ax.axvspan(0, config.angle_tolerance_deg, color='tab:green', alpha=.1)
ax.set(xlabel='Polarity-aware preference error (deg)', ylabel='Density', title='Deep planetary-dominated observations')
ax.legend(frameon=False);

## Ranked candidates

The score rewards sustained expected orientation, small typical error, long planetary episodes, data coverage, lifetime and visible tilt magnitude. Ranking is separate for AE and CE.

In [ ]:
columns = ['Eddy','Cyc','Region','case_score','lifetime_days','regime_observations','regime_fraction','longest_regime_run','matching_fraction','longest_matching_run','median_preference_error_deg','preference_resultant','directional_coverage','median_tilt_km','median_depth_m']
for cyc in ['AE','CE']:
    print(f'\n{cyc} planetary references')
    display(ranking.loc[(ranking.Cyc.eq(cyc)) & ranking.eligible, columns].head(20).round(3))

In [ ]:
selected = select_ranked_cases(ranking, 'Cyc', n_per_group=3)
selected

## Candidate figures

Blue background shading marks qualifying planetary-reference days. Retain one final AE and one final CE after visual inspection; the remaining candidates provide robustness and alternatives.

In [ ]:
for cyc, eddies in selected.items():
    for eddy_id in eddies:
        track = df_case[df_case.Eddy.eq(eddy_id)]
        plot_paper_case(track, grid, config=config, title=f'Planetary reference - {cyc} eddy {eddy_id}')
        plt.show()

## Selected-case audit

Use this table in decision-making, not as inferential statistics. Daily observations within an eddy are repeated measurements.

In [ ]:
selected_ids = [eddy for values in selected.values() for eddy in values]
display(ranking[ranking.Eddy.isin(selected_ids)][columns].sort_values(['Cyc','case_score'], ascending=[True,False]).round(3))

## Interpretation guardrails

A final reference case should remain offshore, have one sustained planetary episode, show adequate tilt magnitude, and maintain the expected polarity-aware direction without relying on a few isolated days. State that the examples illustrate the population result; they do not establish the mechanism independently.